### **Telco Customer Churn Predictor - Encoding and Scaling**

**Objective:** To develop a machine learning classification model that predicts customers who are likely to discontinue their telecom subscription services. The project aims to identify the key factors influencing customer churn through data cleaning, feature engineering, model training, and evaluation, enabling telecom companies to implement targeted customer retention strategies and reduce revenue loss.

**Guiding Principles for this Notebook:**  

+ Load and Inspect data
+ Split Data
+ Encode and Scale


**Import Libraries**

In [1]:
import numpy as np
import pandas as pd
import sys
import os

# Adds the parent directory to the python path
sys.path.append(os.path.abspath(os.path.join("..")))

from sklearn.model_selection import train_test_split
from src.features import (NUMERICAL_FEATURES, CATEGORICAL_FEATURES, TARGET)
from src.preprocessing import create_preprocessor

**Load and Inspect Data**

In [2]:
data = pd.read_csv("C:\\Users\\KOLADE\\OneDrive\\Documents\\AkoladeDSJourney\\Telco-Customer-Churn-Prediction\\data\\processed\\telco_customer_churn_predictors.csv")
df = data.copy()

print(f"Data Shape: {df.shape}")

Data Shape: (7043, 18)


In [3]:
print(f"missing values: {df.isnull().sum().sum()}")
print(f"duplicate values: {df.duplicated().sum()}")
df.head()

missing values: 0
duplicate values: 41


,tenure,Contract,OnlineSecurity,TechSupport,InternetService,PaymentMethod,TotalCharges,OnlineBackup,DeviceProtection,MonthlyCharges,StreamingTV,StreamingMovies,SpendCategory,PaperlessBilling,Dependents,SeniorCitizen,Partner,Churn
0,1,Month-to-month,No,No,DSL,Electronic check,29.85,Yes,No,29.85,No,No,Low,Yes,No,No,Yes,0
1,34,One year,Yes,No,DSL,Mailed check,1889.50,No,Yes,56.95,No,No,Medium,No,No,No,No,0
2,2,Month-to-month,Yes,No,DSL,Mailed check,108.15,Yes,No,53.85,No,No,Medium,Yes,No,No,No,1
3,45,One year,Yes,Yes,DSL,Bank transfer (automatic),1840.75,No,Yes,42.30,No,No,Medium,No,No,No,No,0
4,2,Month-to-month,No,No,Fiber optic,Electronic check,151.65,No,No,70.70,No,No,High,Yes,No,No,No,1


**Train-Test Split**

In [4]:
print(f"Numerical Features: {NUMERICAL_FEATURES}")
print(f"Categorical Features: {CATEGORICAL_FEATURES}")
print(f"Target Feature: {TARGET}")

Numerical Features: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical Features: ['SeniorCitizen', 'Partner', 'Dependents', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SpendCategory']
Target Feature: Churn


In [5]:
X = df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training Set Shape: {X_train.shape}, {y_train.shape}")
print(f"Testing Set Shape: {X_test.shape}, {y_test.shape}")

Training Set Shape: (4930, 17), (4930,)
Testing Set Shape: (2113, 17), (2113,)


In [6]:
print(f"Train_duplicates: {X_train.duplicated().sum()}, Test_duplicates: {X_test.duplicated().sum()}")

Train_duplicates: 30, Test_duplicates: 5


In [7]:
# Check if any rows in the test set exist in the training set
leakage_count = X_test.isin(X_train).all(axis=1).sum()
print(f"Number of leaked rows: {leakage_count}")

Number of leaked rows: 0


**Encode and Scale**

In [8]:
# Create the preprocessor
preprocessor = create_preprocessor()

In [9]:
# Fit the preprocessor on the training data and transform both training and testing data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [10]:
# Convert the processed arrays back to DataFrames
X_train_processed_df = pd.DataFrame(X_train_processed, columns=preprocessor.get_feature_names_out())
X_test_processed_df = pd.DataFrame(X_test_processed, columns=preprocessor.get_feature_names_out())

In [11]:
X_train_processed_df.head()

,num__tenure,num__MonthlyCharges,num__TotalCharges,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__InternetService_Fiber optic,cat__InternetService_No,cat__OnlineSecurity_No internet service,cat__OnlineSecurity_Yes,...,cat__StreamingMovies_Yes,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_Yes,cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__SpendCategory_Low,cat__SpendCategory_Medium,cat__SpendCategory_Premium
0,0.881078,0.195927,0.653962,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1,-1.284263,0.522755,-0.975728,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,-0.793997,-1.509551,-0.896617,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,-0.344587,1.053643,-0.011506,1.0,1.0,1.0,1.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4,-1.079985,0.308740,-0.812139,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


In [13]:
# verify scaling and encoding
X_train_processed_df.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
num__tenure,4930.0,-8.359326e-17,1.000101,-1.325118,-0.957419,-0.140309,0.921933,1.616477
num__MonthlyCharges,4930.0,1.441263e-16,1.000101,-1.544390,-0.975345,0.185973,0.831333,1.785272
num__TotalCharges,4930.0,-1.578183e-16,1.000101,-1.003004,-0.833049,-0.391724,0.671697,2.824909
cat__SeniorCitizen_Yes,4930.0,1.602434e-01,0.366869,0.000000,0.000000,0.000000,0.000000,1.000000
cat__Partner_Yes,4930.0,4.851927e-01,0.499831,0.000000,0.000000,0.000000,1.000000,1.000000


**Variance Inflation Factor (VIF)**

In [14]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [16]:
vif_df = pd.DataFrame()

vif_df["Feature"] = X_train_processed_df.columns

vif_df["VIF"] = [
    variance_inflation_factor(
        X_train_processed_df.values,
        i
    )
    for i in range(X_train_processed_df.shape[1])
]

vif_df = vif_df.sort_values(
    by="VIF",
    ascending=False
)

vif_df

c:\Users\KOLADE\anaconda3\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,Feature,VIF
8,cat__OnlineSecurity_No internet service,inf
10,cat__OnlineBackup_No internet service,inf
12,cat__DeviceProtection_No internet service,inf
7,cat__InternetService_No,inf
14,cat__TechSupport_No internet service,inf
16,cat__StreamingTV_No internet service,inf
18,cat__StreamingMovies_No internet service,inf
1,num__MonthlyCharges,42.063685
26,cat__SpendCategory_Low,21.997925
2,num__TotalCharges,11.401629
